# Section 03.01 — Pandas Fundamentals

**Data Science and Machine Learning Course**

---

## Learning Objectives

By the end of this section you will be able to:
- Create and use `Series` and `DataFrame` objects
- Select columns, rows, and subsets using `loc` and `iloc`
- Filter rows with boolean conditions and multiple criteria
- Reset and set the index
- Work with MultiIndex DataFrames
- Handle missing data with `dropna()` and `fillna()`
- Aggregate with `groupby()`
- Concatenate, merge, and join DataFrames
- Use `apply()`, `value_counts()`, sorting, and pivot tables
- Read and write CSV, Excel, HTML, and SQL

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas: {pd.__version__}")

---

## Part 1 — Series

A `Series` is a one-dimensional labeled array. Think of it as a NumPy array with named indices.

In [ ]:
# Setup data
labels   = ['a', 'b', 'c']
my_data  = [10, 20, 30]
arr      = np.array(my_data)
d        = {'a': 10, 'b': 20, 'c': 30}

In [ ]:
# Series with default integer index
pd.Series(data=my_data)

In [ ]:
# Series with custom labels
pd.Series(data=my_data, index=labels)

In [ ]:
# From a NumPy array
pd.Series(arr, labels)

In [ ]:
# From a dictionary — keys become the index
pd.Series(d)

> A `Series` aligns arithmetic by **index label**, not by position. If labels don't match, the result is `NaN`.

In [ ]:
# Index-aligned arithmetic
ser1 = pd.Series([1, 2, 3, 4], index=['USA', 'Germany', 'USSR', 'Japan'])
ser2 = pd.Series([1, 2, 5, 4], index=['USA', 'Germany', 'Italy', 'Japan'])

ser1 + ser2   # USSR and Italy have no match -> NaN

---

## Part 2 — DataFrames: Creation and Column Operations

A `DataFrame` is a 2D labeled table — essentially a dictionary of `Series` sharing the same index.

In [ ]:
from numpy.random import randn
np.random.seed(101)

df = pd.DataFrame(
    randn(5, 4),
    index=['A', 'B', 'C', 'D', 'E'],
    columns=['W', 'X', 'Y', 'Z']
)

df

In [ ]:
# Select one column -> Series
print(type(df['W']))
df['W']

In [ ]:
# Select multiple columns -> DataFrame
df[['W', 'Z']]

> **Best practice:** use `df['col']` (bracket notation), not `df.col` (dot notation). Dot notation can conflict with built-in DataFrame methods.

In [ ]:
# Add a new column from existing columns
df['new'] = df['W'] + df['Y']
df

In [ ]:
# Drop a column — does NOT modify df unless inplace=True
df.drop('new', axis=1)   # axis=1 -> columns

In [ ]:
# Permanently remove it
df.drop('new', axis=1, inplace=True)

# Drop a row (axis=0)
df.drop('E', axis=0)

**Axis reminder:** `df.shape` returns `(rows, cols)` — rows are axis 0, columns are axis 1.

---

## Part 3 — Row Selection: `loc` and `iloc`

In [ ]:
# loc: label-based
df.loc['C']

In [ ]:
# iloc: integer position-based
df.iloc[2]   # same row, accessed by position

In [ ]:
# Select a single cell: loc[row_label, col_label]
df.loc['B', 'Y']

In [ ]:
# Select multiple rows and columns by label
df.loc[['A', 'B'], ['W', 'Y']]

| Method | Selection basis | Syntax |
|--------|----------------|--------|
| `loc` | Label | `df.loc['B', 'Y']` |
| `iloc` | Integer position | `df.iloc[1, 2]` |

---

## Part 4 — Conditional Filtering

The most important Pandas workflow: filter rows based on column conditions.

In [ ]:
# Boolean mask on the full DataFrame
df > 0

In [ ]:
# Filter on a single column condition -> returns matching rows
df[df['W'] > 0]

In [ ]:
# Chain: filter rows, then select specific columns
df[df['W'] > 0][['Y', 'X']]

# Equivalent expanded form (easier to read while learning):
# mask   = df['W'] > 0
# result = df[mask]
# result[['Y', 'X']]

In [ ]:
# Multiple conditions: use & (and) and | (or)
# IMPORTANT: wrap each condition in parentheses
df[(df['W'] > 0) & (df['Y'] > 1)]

In [ ]:
df[(df['W'] > 0) | (df['Y'] > 1)]

> **Never** use Python's `and` / `or` with Pandas Series conditions — use `&` and `|` instead.

---

## Part 5 — Index Operations

In [ ]:
# reset_index: move current index into a column, assign default integer index
df.reset_index()

In [ ]:
# set_index: promote an existing column to be the index
newind = 'CA NY WY OR CO'.split()
df['States'] = newind
df.set_index('States')

---

## Part 6 — MultiIndex DataFrames

A MultiIndex (hierarchical index) allows rows to be organized by more than one level.

In [ ]:
outside = ['G1', 'G1', 'G1', 'G2', 'G2', 'G2']
inside  = [1, 2, 3, 1, 2, 3]

hier_index = pd.MultiIndex.from_tuples(list(zip(outside, inside)))

df_multi = pd.DataFrame(
    randn(6, 2),
    index=hier_index,
    columns=['A', 'B']
)

df_multi.index.names = ['Groups', 'Num']
df_multi

In [ ]:
# Access the outer level
df_multi.loc['G1']

In [ ]:
# Access a specific inner level value
df_multi.loc['G1'].loc[1]

In [ ]:
# xs: cross-section — slice across a named index level
df_multi.xs(1, level='Num')

---

## Part 7 — Handling Missing Data

In [ ]:
d = {
    'A': [1, 2, np.nan],
    'B': [5, np.nan, np.nan],
    'C': [1, 2, 3]
}

df_miss = pd.DataFrame(d)
df_miss

In [ ]:
# dropna: remove rows (default) or columns with missing values
print(df_miss.dropna())             # drop rows with ANY NaN
print()
print(df_miss.dropna(axis=1))       # drop columns with ANY NaN
print()
print(df_miss.dropna(thresh=2))     # keep rows with at least 2 non-NaN values

In [ ]:
# fillna: replace NaN with a value
print(df_miss.fillna(value=0))
print()

# Fill with column mean — a common imputation strategy
print(df_miss['A'].fillna(value=df_miss['A'].mean()))

> **Note:** The right fill strategy depends on your data. Mean imputation is simple but can distort distributions — consider the context.

---

## Part 8 — GroupBy

`groupby()` splits a DataFrame into groups by a column value, then allows aggregate functions to be applied to each group.

In [ ]:
data = {
    'Company': ['GOOG', 'GOOG', 'MSFT', 'MSFT', 'FB', 'FB'],
    'Person':  ['Sam', 'Charlie', 'Amy', 'Vanessa', 'Carl', 'Sarah'],
    'Sales':   [200, 120, 340, 124, 243, 350]
}

df_sales = pd.DataFrame(data)
df_sales

In [ ]:
# Most common groupby pattern: chain directly
df_sales.groupby('Company').mean(numeric_only=True)

In [ ]:
print(df_sales.groupby('Company').sum(numeric_only=True))
print()
print(df_sales.groupby('Company').count())
print()
print(df_sales.groupby('Company').max(numeric_only=True))

In [ ]:
# describe() gives a full statistical summary per group
df_sales.groupby('Company').describe()

---

## Part 9 — Merging, Joining, and Concatenating

In [ ]:
# Concatenation: stack DataFrames with matching columns
df1 = pd.DataFrame({'A': ['A0','A1','A2'], 'B': ['B0','B1','B2']}, index=[0,1,2])
df2 = pd.DataFrame({'A': ['A3','A4','A5'], 'B': ['B3','B4','B5']}, index=[3,4,5])
df3 = pd.DataFrame({'A': ['A6','A7','A8'], 'B': ['B6','B7','B8']}, index=[6,7,8])

pd.concat([df1, df2, df3])   # axis=0 (rows) by default

In [ ]:
# Merge: SQL-style join on a shared key column
left = pd.DataFrame({
    'key': ['K0','K1','K2','K3'],
    'A':   ['A0','A1','A2','A3'],
    'B':   ['B0','B1','B2','B3']
})

right = pd.DataFrame({
    'key': ['K0','K1','K2','K3'],
    'C':   ['C0','C1','C2','C3'],
    'D':   ['D0','D1','D2','D3']
})

pd.merge(left, right, how='inner', on='key')

In [ ]:
# Merge on multiple keys
left2 = pd.DataFrame({
    'key1': ['K0','K0','K1','K2'], 'key2': ['K0','K1','K0','K1'],
    'A': ['A0','A1','A2','A3'],    'B': ['B0','B1','B2','B3']
})

right2 = pd.DataFrame({
    'key1': ['K0','K1','K1','K2'], 'key2': ['K0','K0','K0','K0'],
    'C': ['C0','C1','C2','C3'],    'D': ['D0','D1','D2','D3']
})

pd.merge(left2, right2, on=['key1', 'key2'])

In [ ]:
# Join: merge on index labels
left3  = pd.DataFrame({'A': ['A0','A1','A2'], 'B': ['B0','B1','B2']}, index=['K0','K1','K2'])
right3 = pd.DataFrame({'C': ['C0','C2','C3'], 'D': ['D0','D2','D3']}, index=['K0','K2','K3'])

left3.join(right3)   # default: left join on index

| Method | Joins on | Use when |
|--------|----------|----------|
| `pd.concat()` | Index alignment | Stacking DataFrames row/column-wise |
| `pd.merge()` | Shared key columns | SQL-style relational joins |
| `df.join()` | Index labels | Combining by index without explicit key |


---

## Part 10 — Common Operations

In [ ]:
df_ops = pd.DataFrame({
    'col1': [1, 2, 3, 4],
    'col2': [444, 555, 666, 444],
    'col3': ['abc', 'def', 'ghi', 'xyz']
})

df_ops

In [ ]:
# unique: distinct values; nunique: count of distinct values
print(df_ops['col2'].unique())
print(df_ops['col2'].nunique())
print(df_ops['col2'].value_counts())

In [ ]:
# apply: call a function on each element of a Series
def times2(x):
    return x * 2

print(df_ops['col1'].apply(times2))
print()
print(df_ops['col3'].apply(len))                    # built-in function
print()
print(df_ops['col2'].apply(lambda x: x * 2))        # inline lambda

In [ ]:
# Sorting
df_ops.sort_values(by='col2')

In [ ]:
# Null check — returns a boolean DataFrame
df_ops.isnull()

In [ ]:
# Pivot table
data_pt = {
    'A': ['foo','foo','foo','bar','bar','bar'],
    'B': ['one','one','two','two','one','one'],
    'C': ['x','y','x','y','x','y'],
    'D': [1, 3, 2, 5, 4, 1]
}

df_pt = pd.DataFrame(data_pt)
df_pt.pivot_table(values='D', index=['A','B'], columns=['C'])

---

## Part 11 — Data Input and Output

Pandas reads and writes many common formats. CSV is the most frequently used.

In [ ]:
# CSV read/write
# df = pd.read_csv('my_data.csv')
# df.to_csv('output.csv', index=False)   # index=False avoids writing row numbers as a column

# Excel read/write
# df = pd.read_excel('data.xlsx', sheet_name='Sheet1')
# df.to_excel('output.xlsx', sheet_name='Results')

# HTML (scrapes all tables on a page)
# tables = pd.read_html('https://example.com/page-with-tables')
# df = tables[0]   # first table found

# SQL (requires sqlalchemy)
# from sqlalchemy import create_engine
# engine = create_engine('sqlite:///:memory:')
# df.to_sql('table_name', engine)
# df = pd.read_sql('table_name', engine)

print("Uncomment the relevant I/O block for your use case.")

| Format | Read | Write |
|--------|------|-------|
| CSV | `pd.read_csv()` | `df.to_csv()` |
| Excel | `pd.read_excel()` | `df.to_excel()` |
| HTML | `pd.read_html()` | — |
| SQL | `pd.read_sql()` | `df.to_sql()` |

> Use `index=False` in `to_csv()` and `to_excel()` when you don't want the DataFrame's index saved as an extra column.

---

## Key Takeaways

- **`Series`** is a labeled 1D array; **`DataFrame`** is a 2D table of Series sharing the same index.
- Use `df['col']` for column selection — never `df.col` in production code.
- `loc[]` selects by label; `iloc[]` selects by integer position.
- Boolean filtering — `df[df['col'] > value]` — is the primary row selection pattern.
- Use `&` and `|` for multiple conditions; wrap each condition in parentheses.
- `groupby().agg()` is the core workflow for split-apply-combine analysis.
- `apply()` enables custom logic column-wise and is essential for feature engineering.
- `merge()` is SQL-style; `concat()` stacks; `join()` uses the index.
- Handle missing data with `dropna()` (remove) or `fillna()` (replace) — choose based on context.

---

*Next: **Section 03.02A — SF Salaries Exercise** and **07B — Ecommerce Exercise** — apply these skills to real datasets.*